In [2]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('../data/real/processed/tb_pheno_geno_clean.csv')

meta_cols = ['isolate', 'glims', 'type', 'lineage']
dlm_cols = ['dlm_mic', 'interp_dlm016', 'interp_dlm06', 'prop_mutants_dlm016', 'prop_mutants_dlm06']
non_mutation_cols = set(meta_cols + dlm_cols + [
    'prop_mutants_bdq12', 'prop_mutants_bdq25', 'prop_mutants_lnz1',
    'interp_bdq12', 'interp_bdq25', 'interp_lnz1',
    'bdq_mic', 'cfz_mic', 'lnz_mic', 'ptm_mic',
])
mutation_cols = [c for c in df.columns if c not in non_mutation_cols]

print(f"Total mutations: {len(mutation_cols)}")
df.head()

Total mutations: 193


,isolate,glims,type,lineage,prop_mutants_bdq12,interp_bdq12,prop_mutants_bdq25,interp_bdq25,prop_mutants_dlm016,interp_dlm016,...,fbid_Val211Gly,fgd1_G*357C,fgd1_Gly120Gly,fgd1_Leu142Leu,fgd1_Lys270Met,fgd1_Phe320Phe,fgd1_T-27G,ddn_C-85A,ddn_Pro124Pro,ddn_Trp88STOP
0,1,2202059251,0,4,0.001,0,0.001,0,0.010,0.0,...,0,0,0,0,0,0,0,0,0,0
1,2,2201006682,0,2,0.001,0,0.001,0,0.001,0.0,...,0,1,0,0,0,1,0,0,0,0
2,3,2201035703,0,4,0.001,0,0.001,0,0.010,0.0,...,0,0,0,0,0,0,0,0,0,0
3,4,2201073436,0,4,0.001,0,0.001,0,0.010,0.0,...,0,0,0,0,1,0,0,0,0,0
4,5,2112064683,0,2,0.001,0,0.001,0,NaN,NaN,...,0,0,0,0,0,1,0,0,0,0


In [4]:
#prevalence filter: keep mutations present in >5% and <98% of patients (data quality, not feature selection)
MIN_PREVALENCE = 0.05
MAX_PREVALENCE = 0.98
min_count = int(np.ceil(MIN_PREVALENCE * len(df)))
max_count = int(np.floor(MAX_PREVALENCE * len(df)))
keep_mutations = [c for c in mutation_cols if min_count <= df[c].sum() <= max_count]
print(f"Mutations with prevalence >{MIN_PREVALENCE*100:.0f}% and <{MAX_PREVALENCE*100:.0f}% (≥{min_count} and ≤{max_count} patients): {len(keep_mutations)}")
print(keep_mutations)

Mutations with prevalence >5% and <98% (≥9 and ≤160 patients): 14
['rv0678_G*128C', 'mmpl5_Asp767Asn', 'mmpl5_Thr794Ile', 'pepq_Ala87Gly', 'pepq_Leu163Arg', 'rv1979c_C*135G', 'rrl_1476188_C>T', 'fbic_C*92G', 'fbic_Thr560Thr', 'fbid_T-298C', 'fbid_Thr207Pro', 'fgd1_G*357C', 'fgd1_Lys270Met', 'fgd1_Phe320Phe']


In [5]:
cispa_cols = ['dlm_mic'] + keep_mutations

print(f"Total variables into CISPA: {len(cispa_cols)}")

df_cispa = df[cispa_cols].dropna()
X = df_cispa.values
print(f"Data shape: {X.shape}")

Total variables into CISPA: 15
Data shape: (164, 15)
